<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 13. 데이터 모달리티와 재사용 가능한 파이프라인

같은 모델링 문제라도 **표형 데이터, 시계열, 텍스트, 이미지**는 샘플의 의미와
올바른 분할 방식이 다릅니다. 이 노트북에서는 외부 다운로드 없이 제공된 소형
데이터로 스키마 검사 → 분할 → 변환 → Dataset/DataLoader까지 연결합니다.

연습본은 `TODO`에서 멈추도록 설계했습니다. 오른쪽 정답본의 같은 번호 셀을
참고하되, 먼저 직접 구현하고 `assert`로 계약을 확인하세요.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 학습 목표

- 행(row), 시간 창(window), 문장, 이미지가 각각 무엇을 한 샘플로 보는지 설명한다.
- 무작위 분할이 시계열/그룹 데이터에서 누수를 만드는 이유를 이해한다.
- `dataclass` 스키마와 상태를 가진 전처리 클래스를 작성한다.
- 길이가 다른 텍스트를 `collate_fn`으로 padding한다.
- NumPy/Pandas와 PyTorch 사이의 dtype·shape 경계를 검사한다.

In [ ]:
import random
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("torch:", torch.__version__)

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "practice").exists():
            return candidate
    raise FileNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. Start_JupyterLab.cmd로 실행해 주세요."
    )


ROOT = find_project_root()
DATA = ROOT / "data" / "practice"
ARTIFACTS = ROOT / "artifacts" / "practice"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

## 1) 네 종류의 데이터를 한 번에 관찰하기

파일 전체를 곧바로 모델에 넣지 말고, 먼저 행 수·열·dtype·결측·레이블 분포를
확인합니다. 텍스트 CSV는 원문 인코딩 상태와 무관하게 `label`과 문자열 길이를
이용할 수 있습니다.

In [ ]:
sensor = pd.read_csv(DATA / "sensor_timeseries.csv", parse_dates=["timestamp"])
tabular = pd.read_csv(DATA / "tabular_risk.csv")
tickets = pd.read_csv(ROOT / "data" / "customer_support_tickets.csv")
shapes = np.load(DATA / "image_shapes.npz")

print("sensor", sensor.shape, sensor.dtypes.to_dict())
print("tabular", tabular.shape, "risk rate=", round(tabular.risk_label.mean(), 3))
print("text", tickets.shape, tickets.label.value_counts().to_dict())
print(
    "image",
    shapes["images"].shape,
    shapes["labels"].shape,
    shapes["class_names"].tolist(),
)

### 모달리티별 샘플 단위

| 모달리티 | 한 샘플 | 보존할 구조 | 대표적인 누수 |
|---|---|---|---|
| 표형 | 계정 한 행 | 열의 의미·범주 | 같은 계정/미래 집계가 양쪽 분할에 존재 |
| 시계열 | 일정 길이의 시간 창 | 시간 순서·장비 그룹 | 미래 값을 학습 통계에 사용 |
| 텍스트 | 문장/문서 | 토큰 순서·길이 | 중복 문서 또는 정답 문구 |
| 이미지 | 채널×높이×너비 | 공간적 이웃 | 원본과 증강본이 서로 다른 분할에 존재 |

## 2) `dataclass`로 스키마 계약 만들기

현업 파이프라인에서는 컬럼명이 조용히 바뀌는 것이 가장 위험한 실패 중 하나입니다.
설정과 검증 규칙을 클래스로 묶으면 테스트와 재사용이 쉬워집니다.

In [ ]:
@dataclass(frozen=True)
class FrameSchema:
    required: tuple[str, ...]
    target: str
    group: str | None = None
    time: str | None = None

    def validate(self, frame: pd.DataFrame) -> None:
        # TODO 13-1: 필수 컬럼, target 결측, time dtype을 검사하세요.
        raise NotImplementedError("오른쪽 정답본을 보기 전에 스키마 검증을 구현하세요.")


SENSOR_SCHEMA = FrameSchema(
    required=(
        "timestamp",
        "device_id",
        "load",
        "temperature",
        "vibration",
        "pressure",
        "anomaly",
    ),
    target="anomaly",
    group="device_id",
    time="timestamp",
)
SENSOR_SCHEMA.validate(sensor)

In [ ]:
assert SENSOR_SCHEMA.target in sensor
assert sensor["device_id"].nunique() >= 2
assert (
    sensor["timestamp"].is_monotonic_increasing is False
)  # 장비별 블록이므로 전체 정렬은 아님
print("schema contract: OK")

## 3) 장비별 시간 분할

각 장비의 앞 70%만 학습, 다음 15%를 검증, 마지막 15%를 테스트로 사용합니다.
이 방식은 모든 장비를 경험하면서도 미래가 과거의 통계에 섞이지 않게 합니다.

In [ ]:
def chronological_group_split(
    frame: pd.DataFrame,
    group_col: str,
    time_col: str,
    train_ratio: float = 0.70,
    valid_ratio: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # TODO 13-2: group별 시간 정렬 후 iloc으로 70/15/15를 나누세요.
    raise NotImplementedError("group-aware chronological split을 구현하세요.")


sensor_train, sensor_valid, sensor_test = chronological_group_split(
    sensor, "device_id", "timestamp"
)

In [ ]:
for device in sensor.device_id.unique():
    tr = sensor_train.query("device_id == @device")
    va = sensor_valid.query("device_id == @device")
    te = sensor_test.query("device_id == @device")
    assert tr.timestamp.max() < va.timestamp.min() < te.timestamp.min()
assert len(sensor_train) + len(sensor_valid) + len(sensor_test) == len(sensor)
print("temporal leakage check: OK")

## 4) 상태를 가진 표형 전처리기

평균·표준편차와 범주 목록은 **train에만 fit**합니다. 검증/테스트에는 저장된 상태를
그대로 적용합니다. 이 작은 클래스는 `sklearn Pipeline`의 `fit/transform` 계약과 같습니다.

In [ ]:
@dataclass
class TabularEncoder:
    numeric: list[str]
    categorical: list[str]
    means_: pd.Series | None = None
    stds_: pd.Series | None = None
    categories_: dict[str, list[str]] | None = None

    def fit(self, frame: pd.DataFrame) -> "TabularEncoder":
        # TODO 13-3a: train의 평균, 표준편차, 범주 목록을 self에 저장하세요.
        raise NotImplementedError

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        # TODO 13-3b: 결측은 train 평균으로 채우고 숫자 표준화 + 범주 one-hot을
        # float32 배열로 합치세요.
        raise NotImplementedError


numeric_cols = [
    "tenure_months",
    "monthly_spend",
    "weekly_sessions",
    "support_tickets_90d",
    "latency_ms",
]
categorical_cols = ["industry", "region", "tier"]
cut = int(len(tabular) * 0.8)
table_train, table_test = tabular.iloc[:cut], tabular.iloc[cut:]
encoder = TabularEncoder(numeric_cols, categorical_cols).fit(table_train)
X_table_train = encoder.transform(table_train)
X_table_test = encoder.transform(table_test)

In [ ]:
assert X_table_train.dtype == np.float32
assert X_table_train.shape[1] == X_table_test.shape[1]
assert np.isfinite(X_table_test).all()
print("tabular feature contract: OK, feature_dim=", X_table_train.shape[1])

## 5) 시계열을 겹치는 window로 변환

`history=12`이면 5분 간격 센서 1시간이 입력 한 개입니다. window가 장비 경계를
넘지 않도록 장비별로 생성하고, 입력은 `(samples, time, features)`가 됩니다.

In [ ]:
SENSOR_FEATURES = ["load", "temperature", "vibration", "pressure"]


def make_windows(frame: pd.DataFrame, history: int = 12, stride: int = 4):
    # TODO 13-4: 장비별로 (history, features) window를 만들고
    # window 안 anomaly의 max를 target으로 반환하세요.
    raise NotImplementedError


X_seq, y_seq = make_windows(sensor_train)

In [ ]:
assert X_seq.ndim == 3 and X_seq.shape[1:] == (12, 4)
assert len(X_seq) == len(y_seq)
assert set(np.unique(y_seq)).issubset({0.0, 1.0})
print("sequence window contract: OK")

## 6) 텍스트 vocabulary와 동적 padding

여기서는 인코딩이 안정적인 짧은 운영 문장을 사용합니다. 실제 프로젝트에서는
정규화/토크나이저 버전과 vocabulary를 artifact로 저장해야 재현할 수 있습니다.

In [ ]:
text_samples = [
    ("payment failed after card renewal", 0),
    ("cancel my subscription before renewal", 1),
    ("login verification code never arrived", 2),
    ("invoice amount is incorrect", 0),
    ("please close the account", 1),
    ("password reset link expired", 2),
]


def tokenize(text: str) -> list[str]:
    return text.lower().replace("?", "").split()


counts = Counter(token for text, _ in text_samples for token in tokenize(text))
stoi = {
    "<pad>": 0,
    "<unk>": 1,
    **{token: i + 2 for i, token in enumerate(sorted(counts))},
}
print("vocab size:", len(stoi), "sample:", tokenize(text_samples[0][0]))

In [ ]:
class TextDataset(Dataset):
    def __init__(self, samples, vocabulary):
        self.samples = samples
        self.vocabulary = vocabulary

    def __len__(self):
        # TODO 13-5a: 샘플 수를 반환하세요.
        raise NotImplementedError

    def __getitem__(self, index):
        # TODO 13-5b: token id LongTensor와 label LongTensor를 반환하세요.
        raise NotImplementedError


def pad_collate(batch):
    # TODO 13-5c: pad_sequence와 원래 lengths를 반환하세요.
    raise NotImplementedError


text_loader = DataLoader(
    TextDataset(text_samples, stoi), batch_size=4, collate_fn=pad_collate
)
token_ids, lengths, labels = next(iter(text_loader))

In [ ]:
assert token_ids.dtype == torch.long and labels.dtype == torch.long
assert token_ids.shape[0] == len(lengths) == len(labels)
assert torch.all((token_ids != 0).sum(dim=1) == lengths)
print("text batch contract: OK")

## 7) 이미지 Dataset과 train 전용 증강

저장 형식은 `(N, H, W)` uint8입니다. 모델 경계에서는 `(C, H, W)` float32와
`[0, 1]` 범위로 바꿉니다. 확률적 증강은 train Dataset에만 적용해야 합니다.

In [ ]:
class ShapeDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, augment: bool = False):
        self.images = images
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # TODO 13-6: (H,W) uint8을 (1,H,W) float32 [0,1]로 바꾸고
        # augment=True일 때 50% 확률 수평 flip을 적용하세요.
        raise NotImplementedError


image_loader = DataLoader(
    ShapeDataset(shapes["images"][:64], shapes["labels"][:64], augment=True),
    batch_size=16,
    shuffle=True,
)
image_batch, image_labels = next(iter(image_loader))

In [ ]:
assert image_batch.shape == (16, 1, 28, 28)
assert image_batch.dtype == torch.float32
assert 0.0 <= image_batch.min() <= image_batch.max() <= 1.0
print("image batch contract: OK")

## 실무 점검표

1. split을 먼저 하고, 통계 fit은 train에서만 했는가?
2. 샘플 단위와 그룹 경계가 명확한가?
3. 모델 직전 dtype과 shape를 `assert`했는가?
4. padding 값과 실제 길이를 함께 전달하는가?
5. 전처리 상태·클래스 순서·seed를 artifact로 남기는가?

In [ ]:
modality_shapes = {
    "tabular": tuple(X_table_train.shape),
    "timeseries": tuple(X_seq.shape),
    "text": tuple(token_ids.shape),
    "image": tuple(image_batch.shape),
}
for name, shape in modality_shapes.items():
    print(f"{name:10s} -> {shape}")
assert all(len(shape) >= 2 for shape in modality_shapes.values())

## 마무리

핵심은 파일 형식이 아니라 **샘플의 의미, 분할 경계, fit 상태, 모델 입력 계약**입니다.
다음 실습에서는 여기서 만든 시계열 원칙을 RNN/GRU/LSTM 분류와 예측에 적용합니다.